In [30]:
from typing import Final

import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [28]:
MIN_N_DATAPOINTS: Final[int] = 2  # need at least one death and one survival
MAX_N_DATAPOINTS: Final[int] = 1_000
MAX_N_FEATURES: Final[int] = 100
N_TRIALS: Final[int] = 100
GROUND_TRUTH_PSURV: Final[float] = 0.50

In [36]:
avg_res_df = pd.DataFrame(
    index=pd.Index(
        name="n_datapoints",
        data=range(MIN_N_DATAPOINTS, MAX_N_DATAPOINTS+1),
    ),
    columns=pd.Index(
        name="n_features",
        # note: first feature will always be `ones`
        data=range(1, MAX_N_FEATURES+1),
    ),
    data=np.nan,
)
avg_res_df

n_features,1,2,3,4,5,6,7,8,9,10,...,91,92,93,94,95,96,97,98,99,100
n_datapoints,,,,,,,,,,,,,,,,,,,,,
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
avg_model_psurv_of_observed_dier_logreg = avg_res_df.copy()
avg_model_psurv_of_observed_survivor_logreg = avg_res_df.copy()
avg_model_psurv_of_observed_dier_tree = avg_res_df.copy()
avg_model_psurv_of_observed_survivor_tree = avg_res_df.copy()

In [ ]:
np.random.seed(42)
rng = np.random.default_rng()

In [ ]:
for n_datapoints in avg_res_df.index:

    # note: first feature will always be `ones`
    for n_features in avg_res_df.columns:

        trialwise_model_psurv_of_observed_dier_logreg = np.empty(shape=N_TRIALS)
        trialwise_model_psurv_of_observed_survivor_logreg = np.empty(shape=N_TRIALS)
        trialwise_model_psurv_of_observed_dier_tree = np.empty(shape=N_TRIALS)
        trialwise_model_psurv_of_observed_survivor_tree = np.empty(shape=N_TRIALS)
        for trial in range(N_TRIALS):

            y = rng.binomial(n=1, p=GROUND_TRUTH_PSURV, size=n_datapoints)
            # hardcode the first two outcomes to be exactly one death and one survival
            y[0] = 0
            y[1] = 1
            X = rng.standard_normal(size=(n_datapoints, n_features))
            # manually account for intercept
            X[:, 0] = 1

            logreg_obj = LogisticRegression(fit_intercept=False)
            logreg_obj.fit(X, y)
            trialwise_model_psurv_of_observed_dier_logreg[trial] = logreg_obj.predict_proba(X[0:1, :])[0, 1]
            trialwise_model_psurv_of_observed_survivor_logreg[trial] = logreg_obj.predict_proba(X[1:2, :])[0, 1]

            tree_obj = DecisionTreeClassifier()
            tree_obj.fit(X, y)
            trialwise_model_psurv_of_observed_dier_tree[trial] = tree_obj.predict_proba(X[0:1, :])[0, 1]
            trialwise_model_psurv_of_observed_survivor_tree[trial] = tree_obj.predict_proba(X[1:2, :])[0, 1]

        avg_model_psurv_of_observed_dier_logreg.loc[n_datapoints, n_features] = trialwise_model_psurv_of_observed_dier_logreg.mean()
        avg_model_psurv_of_observed_survivor_logreg.loc[n_datapoints, n_features] = trialwise_model_psurv_of_observed_survivor_logreg.mean()
        avg_model_psurv_of_observed_dier_tree.loc[n_datapoints, n_features] = trialwise_model_psurv_of_observed_dier_tree.mean()
        avg_model_psurv_of_observed_survivor_tree.loc[n_datapoints, n_features] = trialwise_model_psurv_of_observed_survivor_tree.mean()